# Cross-Factory TTA Experiment

Complete experiment: train YOLOv8m on SH17, evaluate on Pictor-PPE, apply TENT adaptation.

Supports:
- Kaggle (dual T4) and Colab (single GPU) presets
- Auto-resume from checkpoint if interrupted
- All code inline

## Setup

In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "ultralytics", "pyyaml", "pillow", "tqdm"
])

print("Dependencies installed.")

In [ ]:
from pathlib import Path
import os
import shutil
import json
import copy

import torch
import torch.nn as nn
from ultralytics import YOLO
from PIL import Image

print("Imports OK.")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count() if torch.cuda.is_available() else 0}")

## Config

**EDIT THIS SECTION** for your environment:

- **Kaggle**: Set `PLATFORM = "kaggle"`, update dataset paths to `/kaggle/input/datasets/...`
- **Colab**: Set `PLATFORM = "colab"`, update dataset paths to your mounted drive

In [ ]:
# === EDIT THIS SECTION ===

# Platform: "kaggle" or "colab"
PLATFORM = "kaggle"

# Dataset paths (update to your mounts)
if PLATFORM == "kaggle":
    SH17_DIR    = Path("/kaggle/input/datasets/mugheesahmad/sh17-dataset-for-ppe-detection")
    PICTOR_DIR  = Path("/kaggle/input/datasets/zyanahmed/pictor-ppe")
    SHWD_DIR    = Path("/kaggle/input/cross-factory-tta/shwd")      # update slug
    CHV_DIR     = Path("/kaggle/input/cross-factory-tta/chv")       # update slug
    WORK_DIR    = Path("/kaggle/working")
else:  # colab
    SH17_DIR    = Path("/content/drive/MyDrive/datasets/sh17")
    PICTOR_DIR  = Path("/content/drive/MyDrive/datasets/pictor_ppe")
    SHWD_DIR    = Path("/content/drive/MyDrive/datasets/shwd")
    CHV_DIR     = Path("/content/drive/MyDrive/datasets/chv")
    WORK_DIR    = Path("/content")

DATA_DIR = WORK_DIR / "data"
RUNS_DIR = WORK_DIR / "runs"

# === END EDIT SECTION ===

print(f"Platform: {PLATFORM}")
print(f"SH17_DIR exists:   {SH17_DIR.exists()}")
print(f"PICTOR_DIR exists: {PICTOR_DIR.exists()}")
print(f"SHWD_DIR exists:   {SHWD_DIR.exists()}")
print(f"CHV_DIR exists:    {CHV_DIR.exists()}")
print(f"Work dir: {WORK_DIR}")

In [ ]:
# Training hyperparameters (auto-adjusted per platform)

if PLATFORM == "kaggle":
    # Dual T4 preset
    TRAIN_CONFIG = {
        "device": "0,1",
        "batch": 32,
        "workers": 8,
        "lr0": 0.02,  # scaled for larger batch
        "warmup_epochs": 5,
    }
else:
    # Colab single GPU preset (T4/V100/A100)
    TRAIN_CONFIG = {
        "device": 0,
        "batch": 16,
        "workers": 4,
        "lr0": 0.01,
        "warmup_epochs": 3,
    }

# Common hyperparameters
TRAIN_CONFIG.update({
    "epochs": 50,
    "imgsz": 640,
    "optimizer": "SGD",
    "lrf": 0.01,
    "momentum": 0.937,
    "weight_decay": 0.0005,
    "amp": True,
    "box": 7.5,
    "cls": 0.5,
    "dfl": 1.5,
    "hsv_h": 0.015,
    "hsv_s": 0.7,
    "hsv_v": 0.4,
    "translate": 0.1,
    "scale": 0.5,
    "fliplr": 0.5,
    "mosaic": 1.0,
    "verbose": False,
    "exist_ok": True,
})

# SH17 class indices for the three canonical target-domain classes
SH17_EVAL_IDX = {4: "hard_hat", 9: "no_hard_hat", 16: "person"}

# Set to a Path string pointing to an existing best.pt to skip retraining.
# Example: "/kaggle/input/my-sh17-weights/best.pt"
PRETRAINED_WEIGHTS = None

print(f"Training config: batch={TRAIN_CONFIG['batch']}, device={TRAIN_CONFIG['device']}, lr0={TRAIN_CONFIG['lr0']}")
print(f"SH17 eval indices: {SH17_EVAL_IDX}")
print(f"PRETRAINED_WEIGHTS: {PRETRAINED_WEIGHTS}")

## Data Preparation

In [ ]:
SH17_CLASSES = [
    "Coverall", "Face Shield", "Gloves", "Goggles", "Hard Hat",
    "No Coverall", "No Face Shield", "No Gloves", "No Goggles", "No Hard Hat",
    "No Safety Boot", "No Safety Vest", "No Vest", "Safety Boot", "Safety Vest",
    "Vest", "person",
]

CANONICAL_CLASSES = ["hard_hat", "no_hard_hat", "person"]


def setup_sh17():
    """
    Supports two source layouts:

    Layout A — pre-split (our processed upload):
        images/train/  images/val/  labels/train/  labels/val/

    Layout B — public Kaggle dataset (mugheesahmad/sh17-dataset-for-ppe-detection):
        images/  labels/  train_files.txt  val_files.txt
    """
    print("\n=== Setting up SH17 ===")

    layout_a = (SH17_DIR / "images" / "train").exists()
    layout_b = (SH17_DIR / "train_files.txt").exists()

    if not layout_a and not layout_b:
        raise RuntimeError(
            f"[SH17] Unrecognised layout at {SH17_DIR}. "
            "Expected either images/train/ (pre-split) or train_files.txt (public dataset)."
        )

    for split in ("train", "val"):
        out_images = DATA_DIR / "sh17" / "images" / split
        out_labels = DATA_DIR / "sh17" / "labels" / split
        out_images.mkdir(parents=True, exist_ok=True)
        out_labels.mkdir(parents=True, exist_ok=True)

        if layout_a:
            # Pre-split: just copytree
            src_images = SH17_DIR / "images" / split
            src_labels = SH17_DIR / "labels" / split
            if not src_images.exists():
                raise RuntimeError(f"[SH17] images/{split} not found: {src_images}")
            if not src_labels.exists():
                raise RuntimeError(f"[SH17] labels/{split} not found: {src_labels}")
            shutil.copytree(src_images, out_images, dirs_exist_ok=True)
            shutil.copytree(src_labels, out_labels, dirs_exist_ok=True)
        else:
            # Public dataset: read split txt files, copy file by file
            split_file = SH17_DIR / f"{split}_files.txt"
            filenames = [f.strip() for f in split_file.read_text().splitlines() if f.strip()]
            for fname in filenames:
                src_img = SH17_DIR / "images" / fname
                src_lbl = SH17_DIR / "labels" / (Path(fname).stem + ".txt")
                if src_img.exists():
                    shutil.copy2(src_img, out_images / fname)
                if src_lbl.exists():
                    shutil.copy2(src_lbl, out_labels / src_lbl.name)

        n = len(list(out_images.glob("*")))
        print(f"  SH17 {split}: {n} images")


def _copy_split(src_dir: Path, out_dir: Path, label: str):
    src_images = src_dir / "images" / "test"
    src_labels = src_dir / "labels" / "test"

    if not src_images.exists():
        raise RuntimeError(f"[{label}] images not found: {src_images}")
    if not src_labels.exists():
        raise RuntimeError(f"[{label}] labels not found: {src_labels}")

    out_images = out_dir / "images" / "test"
    out_labels = out_dir / "labels" / "test"
    out_images.mkdir(parents=True, exist_ok=True)
    out_labels.mkdir(parents=True, exist_ok=True)

    shutil.copytree(src_images, out_images, dirs_exist_ok=True)
    shutil.copytree(src_labels, out_labels, dirs_exist_ok=True)
    print(f"  {label} test: {len(list(out_images.glob('*')))} images")


def setup_pictor():
    print("\n=== Setting up Pictor-PPE ===")
    _copy_split(PICTOR_DIR, DATA_DIR / "pictor_ppe", "Pictor")


def setup_shwd():
    print("\n=== Setting up SHWD ===")
    _copy_split(SHWD_DIR, DATA_DIR / "shwd", "SHWD")


def setup_chv():
    print("\n=== Setting up CHV ===")
    _copy_split(CHV_DIR, DATA_DIR / "chv", "CHV")


def write_yamls():
    print("\n=== Writing YAML configs ===")

    (DATA_DIR / "sh17" / "sh17.yaml").write_text(f"""path: {DATA_DIR / 'sh17'}
train: images/train
val: images/val

nc: {len(SH17_CLASSES)}
names: {SH17_CLASSES}
""")

    for ds_name, yaml_name in [
        ("pictor_ppe", "pictor.yaml"),
        ("shwd",       "shwd.yaml"),
        ("chv",        "chv.yaml"),
    ]:
        (DATA_DIR / ds_name / yaml_name).write_text(f"""path: {DATA_DIR / ds_name}
train: images/test
val: images/test

nc: {len(CANONICAL_CLASSES)}
names: {CANONICAL_CLASSES}
""")
        print(f"  {yaml_name} written")


def _data_ready() -> bool:
    checks = [
        DATA_DIR / "sh17"       / "sh17.yaml",
        DATA_DIR / "pictor_ppe" / "pictor.yaml",
        DATA_DIR / "shwd"       / "shwd.yaml",
        DATA_DIR / "chv"        / "chv.yaml",
    ]
    missing = [str(p) for p in checks if not p.exists()]
    if missing:
        print(f"Missing: {missing}")
    return len(missing) == 0


if _data_ready():
    print("Data already prepared.")
else:
    setup_sh17()
    setup_pictor()
    setup_shwd()
    setup_chv()
    write_yamls()
    print("\nData preparation complete.")

## Training on SH17

Auto-resumes from checkpoint if available.

In [ ]:
RUN_NAME          = "sh17_yolov8m"
CHECKPOINT_PATH   = RUNS_DIR / "train" / RUN_NAME / "weights" / "last.pt"
BEST_WEIGHTS_PATH = RUNS_DIR / "train" / RUN_NAME / "weights" / "best.pt"

# Fix 1: Load pre-trained weights if provided — skips training entirely
if PRETRAINED_WEIGHTS is not None:
    src = Path(PRETRAINED_WEIGHTS)
    if not src.exists():
        raise RuntimeError(f"PRETRAINED_WEIGHTS set but file not found: {src}")
    BEST_WEIGHTS_PATH.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, BEST_WEIGHTS_PATH)
    print(f"Loaded pre-trained weights from {src}, skipping training.")

# Fix 5: Resume interrupted training from last.pt (same session only)
elif CHECKPOINT_PATH.exists():
    print(f"Resuming interrupted training from {CHECKPOINT_PATH}")
    model = YOLO(str(CHECKPOINT_PATH))
    results = model.train(resume=True)
    print(f"\nTraining complete. Best weights: {BEST_WEIGHTS_PATH}")

else:
    # Fresh training — need to download yolov8m.pt base weights
    # Fix 2: Fail fast if internet is off and base weights are missing
    def _internet_available() -> bool:
        try:
            import urllib.request
            urllib.request.urlopen("https://ultralytics.com", timeout=5)
            return True
        except Exception:
            return False

    if not Path("yolov8m.pt").exists() and not _internet_available():
        raise RuntimeError(
            "Internet is disabled and no local yolov8m.pt found. "
            "Either enable Kaggle internet or attach pre-trained weights via PRETRAINED_WEIGHTS."
        )

    def compact_epoch_callback(trainer):
        try:
            loss_items = getattr(trainer, "loss_items", None)
            box = cls = dfl = None
            if loss_items is not None and hasattr(loss_items, "__len__") and len(loss_items) >= 3:
                box, cls, dfl = float(loss_items[0]), float(loss_items[1]), float(loss_items[2])

            metrics = getattr(trainer, "metrics", {})
            if not isinstance(metrics, dict):
                metrics = getattr(metrics, "results_dict", {})

            map50   = metrics.get("metrics/mAP50(B)")    or metrics.get("metrics/mAP50")
            map5095 = metrics.get("metrics/mAP50-95(B)") or metrics.get("metrics/mAP50-95")

            epoch  = int(getattr(trainer, "epoch",  -1)) + 1
            epochs = int(getattr(trainer, "epochs", -1))

            parts = [f"epoch {epoch}/{epochs}"]
            if box     is not None: parts.append(f"box={box:.3f}")
            if cls     is not None: parts.append(f"cls={cls:.3f}")
            if dfl     is not None: parts.append(f"dfl={dfl:.3f}")
            if map50   is not None: parts.append(f"mAP50={float(map50):.3f}")
            if map5095 is not None: parts.append(f"mAP50-95={float(map5095):.3f}")

            print(" | ".join(parts))
        except Exception as e:
            print(f"[WARN] Callback error: {e}")

    print("epoch/epochs | box | cls | dfl | mAP50 | mAP50-95")

    model = YOLO("yolov8m.pt")
    model.add_callback("on_fit_epoch_end", compact_epoch_callback)

    results = model.train(
        data=str(DATA_DIR / "sh17" / "sh17.yaml"),
        name=RUN_NAME,
        project=str(RUNS_DIR / "train"),
        **TRAIN_CONFIG
    )

    print(f"\nTraining complete.")
    print(f"Best weights saved to: {BEST_WEIGHTS_PATH}")
    print("→ Download best.pt and set PRETRAINED_WEIGHTS to skip retraining next session.")
    if BEST_WEIGHTS_PATH.exists():
        print(f"SH17 val mAP50:    {results.results_dict.get('metrics/mAP50(B)', 'N/A')}")
        print(f"SH17 val mAP50-95: {results.results_dict.get('metrics/mAP50-95(B)', 'N/A')}")

In [ ]:
import shutil as _shutil

print("=" * 55)
print("PREFLIGHT CHECK")
print("=" * 55)

_ok = True

# CUDA
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f"[OK] CUDA available — {gpu_name} (x{torch.cuda.device_count()})")
else:
    print("[FAIL] No CUDA — training will be extremely slow")
    _ok = False

# Ultralytics version
try:
    import ultralytics
    print(f"[OK] ultralytics {ultralytics.__version__}")
except Exception as e:
    print(f"[FAIL] ultralytics not importable: {e}")
    _ok = False

# Disk space (warn if < 10 GB free in WORK_DIR)
_stat = _shutil.disk_usage(str(WORK_DIR))
_free_gb = _stat.free / 1e9
if _free_gb < 10:
    print(f"[WARN] Only {_free_gb:.1f} GB free in {WORK_DIR} — may run out during training")
else:
    print(f"[OK] {_free_gb:.1f} GB free in {WORK_DIR}")

# Dataset directories and counts
_expected = {
    "sh17":       {"train": 6479, "val": 1620},
    "shwd":       {"test":  1517},
    "pictor_ppe": {"test":  152},
    "chv":        {"test":  133},
}
for ds, splits in _expected.items():
    for split, expected_count in splits.items():
        img_dir = DATA_DIR / ds / "images" / split
        lbl_dir = DATA_DIR / ds / "labels" / split
        if not img_dir.exists():
            print(f"[FAIL] {ds}/{split} images dir missing: {img_dir}")
            _ok = False
            continue
        n_imgs = len(list(img_dir.glob("*")))
        n_lbls = len(list(lbl_dir.glob("*.txt"))) if lbl_dir.exists() else 0
        status = "OK" if n_imgs == expected_count else "WARN"
        print(f"[{status}] {ds}/{split}: {n_imgs} images, {n_lbls} labels (expected {expected_count})")
        if n_imgs != expected_count:
            _ok = False

# YAML files and nc values
_yaml_checks = [
    (DATA_DIR / "sh17"       / "sh17.yaml",    17),
    (DATA_DIR / "pictor_ppe" / "pictor.yaml",   3),
    (DATA_DIR / "shwd"       / "shwd.yaml",     3),
    (DATA_DIR / "chv"        / "chv.yaml",      3),
]
import yaml as _yaml
for yaml_path, expected_nc in _yaml_checks:
    if not yaml_path.exists():
        print(f"[FAIL] Missing yaml: {yaml_path}")
        _ok = False
        continue
    nc = _yaml.safe_load(yaml_path.read_text()).get("nc")
    if nc != expected_nc:
        print(f"[FAIL] {yaml_path.name}: nc={nc}, expected {expected_nc}")
        _ok = False
    else:
        print(f"[OK] {yaml_path.name}: nc={nc}")

# Weights
if PRETRAINED_WEIGHTS is not None:
    p = Path(PRETRAINED_WEIGHTS)
    if p.exists():
        print(f"[OK] PRETRAINED_WEIGHTS found: {p}")
    else:
        print(f"[FAIL] PRETRAINED_WEIGHTS set but file missing: {p}")
        _ok = False
else:
    print("[INFO] PRETRAINED_WEIGHTS=None — will train from scratch")

print("=" * 55)
if not _ok:
    raise RuntimeError("Preflight failed — fix errors above before proceeding.")
print("All checks passed. Ready to run.")

In [ ]:
def make_remapped_yaml(ds_name: str, tmp_root: Path) -> Path:
    """
    Remap canonical label indices [0,1,2] → SH17 indices [4,9,16] and write
    a temp yaml so .val() matches predictions at the correct class slots.

        0 (hard_hat)    → 4  (Hard Hat in SH17)
        1 (no_hard_hat) → 9  (No Hard Hat in SH17)
        2 (person)      → 16 (person in SH17)

    Images are copied via shutil.copy2 (symlinks unreliable on Kaggle).
    Caller is responsible for cleanup via shutil.rmtree.
    """
    CANONICAL_TO_SH17 = {0: 4, 1: 9, 2: 16}

    tmp_dir    = tmp_root / ds_name
    tmp_labels = tmp_dir / "labels" / "test"
    tmp_images = tmp_dir / "images" / "test"
    tmp_labels.mkdir(parents=True, exist_ok=True)
    tmp_images.mkdir(parents=True, exist_ok=True)

    src_images = DATA_DIR / ds_name / "images" / "test"
    for img in src_images.iterdir():
        shutil.copy2(img, tmp_images / img.name)

    src_labels = DATA_DIR / ds_name / "labels" / "test"
    for lbl in src_labels.glob("*.txt"):
        lines_out = []
        for line in lbl.read_text().splitlines():
            parts = line.split()
            if not parts:
                continue
            cls = int(parts[0])
            lines_out.append(f"{CANONICAL_TO_SH17[cls]} {' '.join(parts[1:])}")
        (tmp_labels / lbl.name).write_text(
            "\n".join(lines_out) + "\n" if lines_out else ""
        )

    tmp_yaml = tmp_dir / f"{ds_name}_remap.yaml"
    tmp_yaml.write_text(f"""path: {tmp_dir}
train: images/test
val: images/test

nc: {len(SH17_CLASSES)}
names: {SH17_CLASSES}
""")
    return tmp_yaml


print("make_remapped_yaml ready.")

## Baseline Evaluation (Zero-Shot)

In [ ]:
print("\n=== Baseline Zero-Shot Evaluation (All Target Domains) ===")

TMP_ROOT = WORK_DIR / "tmp_remap"

TARGET_DOMAINS = [
    ("pictor_ppe", "pictor.yaml"),
    ("shwd",       "shwd.yaml"),
    ("chv",        "chv.yaml"),
]

baseline_model = YOLO(str(BEST_WEIGHTS_PATH))
baseline_results = {}

for ds_name, _ in TARGET_DOMAINS:
    tmp_yaml = make_remapped_yaml(ds_name, TMP_ROOT)

    metrics = baseline_model.val(
        data=str(tmp_yaml),
        imgsz=640,
        batch=16,
        split="val",
        conf=0.001,
        iou=0.6,
        name=f"baseline_{ds_name}",
        project=str(RUNS_DIR / "eval"),
        exist_ok=True,
        verbose=False,
    )

    # ap_class_index lists classes that had GT boxes (Ultralytics >= 8.0).
    # Fall back to range(len(ap50)) and filter to {4,9,16} manually if absent.
    _class_idx = (
        metrics.box.ap_class_index
        if hasattr(metrics.box, "ap_class_index")
        else range(len(metrics.box.ap50))
    )

    per_class_ap50, per_class_ap5095 = {}, {}
    for idx, ap50, ap5095 in zip(_class_idx, metrics.box.ap50, metrics.box.ap):
        if idx in SH17_EVAL_IDX:
            name = SH17_EVAL_IDX[idx]
            per_class_ap50[name]   = float(ap50)
            per_class_ap5095[name] = float(ap5095)

    mean_map50   = sum(per_class_ap50.values())   / len(per_class_ap50)   if per_class_ap50   else 0.0
    mean_map5095 = sum(per_class_ap5095.values()) / len(per_class_ap5095) if per_class_ap5095 else 0.0

    baseline_results[ds_name] = {
        "mAP50":          mean_map50,
        "mAP50_95":       mean_map5095,
        "per_class_AP50": per_class_ap50,
    }
    print(f"  {ds_name:<14}  mAP50={mean_map50:.4f}  mAP50-95={mean_map5095:.4f}  "
          f"classes={list(per_class_ap50.keys())}")
    shutil.rmtree(TMP_ROOT / ds_name, ignore_errors=True)

# Preserve pictor scalars — TENT cell references these
baseline_map50   = baseline_results["pictor_ppe"]["mAP50"]
baseline_map5095 = baseline_results["pictor_ppe"]["mAP50_95"]

print(f"\n{'Dataset':<14} {'mAP50':>8} {'mAP50-95':>10}")
print("-" * 34)
for ds, res in baseline_results.items():
    print(f"{ds:<14} {res['mAP50']:>8.4f} {res['mAP50_95']:>10.4f}")

del baseline_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# TENT implementation

def configure_model_for_tent(model: nn.Module) -> nn.Module:
    model.train()
    for param in model.parameters():
        param.requires_grad_(False)
    for m in model.modules():
        if isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
            m.requires_grad_(True)
            m.track_running_stats = False
            m.running_mean = None
            m.running_var = None
    return model


def collect_bn_params(model: nn.Module):
    params, names = [], []
    for nm, m in model.named_modules():
        if isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
            for pnm, p in m.named_parameters():
                if p.requires_grad:
                    params.append(p)
                    names.append(f"{nm}.{pnm}")
    return params, names


def softmax_entropy(logits: torch.Tensor) -> torch.Tensor:
    probs = logits.softmax(dim=-1)
    return -(probs * (probs + 1e-8).log()).sum(dim=-1)


def tent_loss(predictions) -> torch.Tensor:
    if isinstance(predictions, (list, tuple)):
        entropy_terms = []
        for pred in predictions:
            if not isinstance(pred, torch.Tensor):
                continue
            if pred.ndim == 3 and pred.shape[1] > 4:
                cls_logits = pred[:, 4:, :].permute(0, 2, 1)
                entropy_terms.append(softmax_entropy(cls_logits))
        if entropy_terms:
            return torch.cat([e.reshape(-1) for e in entropy_terms]).mean()
    return torch.tensor(0.0, requires_grad=True)


class TENT:
    def __init__(self, model: YOLO, lr: float = 0.001, steps: int = 1):
        self.model = model
        self.steps = steps
        self.lr    = lr
        self._original_state = copy.deepcopy(model.model.state_dict())
        configure_model_for_tent(model.model)
        params, _ = collect_bn_params(model.model)
        print(f"[TENT] Adapting {len(params)} BN parameters")
        self.optimizer = torch.optim.SGD(params, lr=lr, momentum=0.9)

    @torch.enable_grad()
    def step(self, batch_paths: list) -> list:
        for _ in range(self.steps):
            results_raw = self.model.model(self._preprocess(batch_paths))
            loss = tent_loss(results_raw)
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
        return self.model.predict(batch_paths, verbose=False)

    def _preprocess(self, paths: list) -> torch.Tensor:
        from ultralytics.data.augment import LetterBox
        import cv2
        tensors = []
        lb = LetterBox(new_shape=(640, 640))
        for p in paths:
            img = cv2.imread(str(p))
            img = lb(image=img)
            img = img[:, :, ::-1].transpose(2, 0, 1).copy()
            tensors.append(torch.from_numpy(img).float() / 255.0)
        batch = torch.stack(tensors)
        return batch.to(next(self.model.model.parameters()).device)


print("TENT implementation ready.")

print("\n" + "=" * 55)
print("FINAL RESULTS")
print("=" * 55)

# Source domain: evaluate on SH17 val with the trained model
print("\nEvaluating source domain (SH17 val)...")
_src_model = YOLO(str(BEST_WEIGHTS_PATH))
_src_metrics = _src_model.val(
    data=str(DATA_DIR / "sh17" / "sh17.yaml"),
    imgsz=640,
    batch=16,
    split="val",
    conf=0.001,
    iou=0.6,
    name="source_sh17",
    project=str(RUNS_DIR / "eval"),
    exist_ok=True,
    verbose=False,
)
sh17_map50   = float(_src_metrics.box.map50)
sh17_map5095 = float(_src_metrics.box.map)
del _src_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Print summary table
print(f"\nSource domain (SH17 val):    mAP50={sh17_map50:.4f}  mAP50-95={sh17_map5095:.4f}")
print(f"\nTarget domain zero-shot baseline:")
for ds, res in baseline_results.items():
    print(f"  {ds:<14}  mAP50={res['mAP50']:.4f}  mAP50-95={res['mAP50_95']:.4f}")
print(f"\nTENT adaptation (pictor_ppe):")
print(f"  mAP50={tent_map50:.4f}  mAP50-95={tent_map5095:.4f}  recovery={recovery:+.4f}")
print("=" * 55)

# Save results JSON
final_results = {
    "platform": PLATFORM,
    "training_config": TRAIN_CONFIG,
    "source_sh17": {
        "mAP50":    sh17_map50,
        "mAP50_95": sh17_map5095,
    },
    "baseline": {
        ds: {"mAP50": res["mAP50"], "mAP50_95": res["mAP50_95"]}
        for ds, res in baseline_results.items()
    },
    "tent_pictor": {
        "mAP50":          tent_map50,
        "mAP50_95":       tent_map5095,
        "recovery_mAP50": recovery,
    },
    "per_class_AP50": {
        "baseline":    {ds: res["per_class_AP50"] for ds, res in baseline_results.items()},
        "tent_pictor": tent_per_class_ap50,
    },
}

results_path = WORK_DIR / "results.json"
results_path.write_text(json.dumps(final_results, indent=2))
print(f"\nSaved to: {results_path}")

In [ ]:
print("\n=== TENT Adaptation on Pictor ===")

tmp_yaml_pictor = make_remapped_yaml("pictor_ppe", TMP_ROOT)

tent_model = YOLO(str(BEST_WEIGHTS_PATH))
adapter = TENT(tent_model, lr=0.001, steps=1)

def on_val_batch_start(validator):
    batch_imgs = validator.batch.get("im_file", [])
    if batch_imgs:
        adapter.step(list(batch_imgs))

tent_model.add_callback("on_val_batch_start", on_val_batch_start)

tent_metrics = tent_model.val(
    data=str(tmp_yaml_pictor),
    imgsz=640,
    batch=16,
    split="val",
    conf=0.001,
    iou=0.6,
    name="tent_pictor",
    project=str(RUNS_DIR / "eval"),
    exist_ok=True,
    verbose=False,
)

_class_idx = (
    tent_metrics.box.ap_class_index
    if hasattr(tent_metrics.box, "ap_class_index")
    else range(len(tent_metrics.box.ap50))
)

tent_per_class_ap50, tent_per_class_ap5095 = {}, {}
for idx, ap50, ap5095 in zip(_class_idx, tent_metrics.box.ap50, tent_metrics.box.ap):
    if idx in SH17_EVAL_IDX:
        name = SH17_EVAL_IDX[idx]
        tent_per_class_ap50[name]   = float(ap50)
        tent_per_class_ap5095[name] = float(ap5095)

tent_map50   = sum(tent_per_class_ap50.values())   / len(tent_per_class_ap50)   if tent_per_class_ap50   else 0.0
tent_map5095 = sum(tent_per_class_ap5095.values()) / len(tent_per_class_ap5095) if tent_per_class_ap5095 else 0.0
recovery     = tent_map50 - baseline_map50

shutil.rmtree(TMP_ROOT / "pictor_ppe", ignore_errors=True)

print(f"TENT mAP50:    {tent_map50:.4f}")
print(f"TENT mAP50-95: {tent_map5095:.4f}")
print(f"Recovery:      {recovery:+.4f}")

In [ ]:
print("\n" + "="*60)
print("FINAL RESULTS")
print("="*60)

print(f"\n{'Dataset':<14} {'Baseline mAP50':>16} {'Baseline mAP50-95':>18}")
print("-" * 50)
for ds_name, res in baseline_results.items():
    print(f"{ds_name:<14} {res['mAP50']:>16.4f} {res['mAP50_95']:>18.4f}")

print(f"\nTENT (Pictor-PPE only):")
print(f"  Baseline mAP50:  {baseline_map50:.4f}")
print(f"  TENT mAP50:      {tent_map50:.4f}")
print(f"  Recovery:        {recovery:+.4f}")
print("="*60)

final_results = {
    "platform": PLATFORM,
    "training_config": TRAIN_CONFIG,
    "baseline": {
        ds_name: {
            "mAP50":    res["mAP50"],
            "mAP50_95": res["mAP50_95"],
        }
        for ds_name, res in baseline_results.items()
    },
    "tent_pictor": {
        "mAP50":    tent_map50,
        "mAP50_95": tent_map5095,
        "recovery_mAP50": recovery,
    },
    "per_class_AP50": {
        "baseline": {ds: res["per_class_AP50"] for ds, res in baseline_results.items()},
        "tent_pictor": {
            name: float(ap)
            for name, ap in zip(tent_metrics.names.values(), tent_metrics.box.ap50)
        },
    },
}

results_path = WORK_DIR / f"results_{PLATFORM}.json"
results_path.write_text(json.dumps(final_results, indent=2))
print(f"\nSaved to: {results_path}")